# 🔍 Ignite Data Lake — Schema Validator

**Purpose:** S3 (via Trino) se fetch kiya hua data, Postgres schema (CSV export) ke against validate karna.

| File | Source |
|------|--------|
| **Schema** | Postgres/RDS → DataGrip CSV export |
| **Data** | S3 bucket → Trino query |

### Supported Data Types (Schema mein)
| Schema Type | Validates As |
|-------------|-------------|
| `STRING` | Any non-empty text |
| `BIGINT` | Whole number (no decimals) |
| `BOOLEAN` | true/false/1/0/yes/no |
| `DOUBLE` | Any numeric value (int or decimal) |
| `DECIMAL`, `DECIMAL(p,s)` | High-precision numeric |
| `DATE` | Parseable date string |
| `TIMESTAMP` | Parseable datetime string |
| `ARRAY<STRING>` | JSON list of strings e.g. `["abc", "xyz"]` |

---
## 📦 Cell 1 — Libraries Install

In [7]:
!pip install pandas openpyxl xlrd trino -q

import re
import ast
import urllib3
import pandas as pd
import numpy as np
from decimal import Decimal, InvalidOperation, getcontext
from IPython.display import display
from trino.dbapi import connect
from trino.auth import BasicAuthentication

# DECIMAL(38,18) ke liye precision set karo
getcontext().prec = 38

# SSL warnings chupao (self-signed cert)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

print("✅ Libraries loaded!")

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

---
## ⚙️ Cell 2 — Configuration (Sirf yahan change karo)

In [8]:
# ─────────────────────────────────────────────────────────────────
#  SCHEMA FILE (Postgres → DataGrip CSV export)
# ─────────────────────────────────────────────────────────────────
SCHEMA_FILE          = "/home/shariq/Downloads/data_configurator_puite_audittrail_schema.csv"
DATATYPE_COLUMN_NAME = "datatype"    # Schema CSV mein datatype column ka naam
ID_COLUMN_NAME       = "id"          # Schema CSV mein id column ka naam

# ─────────────────────────────────────────────────────────────────
#  TRINO CONNECTION
# ─────────────────────────────────────────────────────────────────
TRINO_CONFIG = {
   "host": "3.221.185.123",  # e.g., "trino.ignite.com"
    "port": 8443,
    "user": "shariq.mehmood",
    "password": "Prometheus-X!",
    "catalog": "iceberg",
    "schema": "ignite_test"
}

# ─────────────────────────────────────────────────────────────────
#  QUERY INFO
# ─────────────────────────────────────────────────────────────────
TARGET_TABLE = "iceberg.ignite_test.silver_audittrail "
S3_LOG_PATH  = "s3a://kuza-ignite/bronze/AuditTrailDAS/AWS/Primary/Sao_Paulo/AuditTrailDAS/Request-22-04-2026.log"

print(f"✅ Schema File   : {SCHEMA_FILE}")
print(f"✅ Target Table  : {TARGET_TABLE}")
print(f"✅ S3 Log Path   : {S3_LOG_PATH}")

✅ Schema File   : /home/shariq/Downloads/data_configurator_puite_audittrail_schema.csv
✅ Target Table  : iceberg.ignite_test.silver_audittrail 
✅ S3 Log Path   : s3a://kuza-ignite/bronze/AuditTrailDAS/AWS/Primary/Sao_Paulo/AuditTrailDAS/Request-22-04-2026.log


---
## 🔌 Cell 3 — Trino se Data Fetch karo

In [9]:
def get_data_from_trino() -> pd.DataFrame:
    try:
        auth = BasicAuthentication(TRINO_CONFIG["user"], TRINO_CONFIG["password"])
        conn = connect(
            host       = TRINO_CONFIG["host"],
            port       = TRINO_CONFIG["port"],
            auth       = auth,
            http_scheme= "https",
            verify     = False,
            catalog    = TRINO_CONFIG["catalog"],
            schema     = TRINO_CONFIG["schema"]
        )
        query = f"""
            SELECT *
            FROM {TARGET_TABLE}
            WHERE s3_path = '{S3_LOG_PATH}'
        """
        print("⏳ Trino se data fetch ho raha hai (HTTPS, SSL verify=False)...")
        cur = conn.cursor()
        cur.execute(query)
        rows = cur.fetchall()
        cols = [desc[0] for desc in cur.description]
        df   = pd.DataFrame(rows, columns=cols)
        # Sabko string mein convert karo — type checking manually hogi
        df   = df.astype(str).replace(["None", "nan", "NaN"], "")
        print(f"✅ Data fetch ho gaya! Total rows: {len(df)}, Columns: {len(df.columns)}")
        return df
    except Exception as e:
        print(f"❌ Trino connection failed: {e}")
        return pd.DataFrame()

data_df = get_data_from_trino()

⏳ Trino se data fetch ho raha hai (HTTPS, SSL verify=False)...
✅ Data fetch ho gaya! Total rows: 439548, Columns: 144


---
## 📚 Cell 4 — Type Mapping aur Validation Functions

Yahan har schema data type ko properly handle kiya gaya hai:

- **`STRING`** → koi bhi non-empty value valid
- **`BIGINT`** → whole numbers only (`42`, `-5`, `1000`), decimal allowed nahi (`42.5` ❌)
- **`DOUBLE`** → koi bhi number (`42`, `3.14`, `-0.001`, scientific notation)
- **`DECIMAL / DECIMAL(p,s)`** → high-precision numeric via Python `Decimal`
- **`BOOLEAN`** → `true/false/1/0/yes/no` (case-insensitive)
- **`DATE`** → pandas parseable date string
- **`TIMESTAMP`** → pandas parseable datetime string
- **`ARRAY<STRING>`** → valid JSON list jisme har element ek string ho

In [10]:
# ─── Type Alias Map ───────────────────────────────────────────────────────────
# Schema mein jo bhi type aaye, usse ek base category mein map karo
TYPE_ALIASES = {
    # BIGINT family — whole numbers only
    "bigint"   : "bigint",
    "int"      : "bigint",
    "integer"  : "bigint",
    "int32"    : "bigint",
    "int64"    : "bigint",
    "smallint" : "bigint",
    "tinyint"  : "bigint",
 
    # DOUBLE family — any numeric
    "double"   : "double",
    "float"    : "double",
    "float32"  : "double",
    "float64"  : "double",
    "real"     : "double",
 
    # DECIMAL family — high precision (handles DECIMAL(38,18) etc)
    "decimal"  : "decimal",
    "numeric"  : "decimal",
    "number"   : "decimal",
 
    # STRING family
    "string"   : "string",
    "str"      : "string",
    "varchar"  : "string",
    "char"     : "string",
    "text"     : "string",
    "nvarchar" : "string",
    "nchar"    : "string",
 
    # BOOLEAN
    "boolean"  : "boolean",
    "bool"     : "boolean",
 
    # DATE
    "date"     : "date",
 
    # TIMESTAMP family
    "timestamp": "timestamp",
    "datetime" : "timestamp",
    "time"     : "timestamp",
 
    # ARRAY<STRING>
    "array<string>": "array_string",
}
 
 
def normalize_dtype(raw: str) -> str:
    """
    Schema ki raw datatype string ko base category mein convert karo.
 
    Examples:
        'DECIMAL(38,18)' -> 'decimal'
        'ARRAY<STRING>'  -> 'array_string'
        'BIGINT'         -> 'bigint'
        'DOUBLE'         -> 'double'
        'STRING'         -> 'string'
        'TIMESTAMP'      -> 'timestamp'
    """
    cleaned = str(raw).strip().lower()
 
    # ARRAY<STRING> — pehle check karo (parentheses strip karne se pehle)
    if "array<string>" in cleaned:
        return "array_string"
 
    # DECIMAL with precision e.g. decimal(38,18) -> decimal
    if "decimal" in cleaned:
        return "decimal"
 
    # Parentheses wala part hatao: varchar(255) -> varchar
    base = re.sub(r"\(.*?\)", "", cleaned).strip()
 
    return TYPE_ALIASES.get(base, None)  # None = unknown type
 
 
# ─── Utility Functions ────────────────────────────────────────────────────────
 
def read_file(filepath: str) -> pd.DataFrame:
    """CSV ya Excel file padhna — sab string mein."""
    lower  = filepath.lower()
    kwargs = {"dtype": str, "keep_default_na": False}
    if lower.endswith(".csv")   : return pd.read_csv(filepath, **kwargs)
    elif lower.endswith(".xlsx"): return pd.read_excel(filepath, engine="openpyxl", dtype=str)
    elif lower.endswith(".xls") : return pd.read_excel(filepath, engine="xlrd", dtype=str)
    else                        : return pd.read_csv(filepath, **kwargs)
 
 
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Column names lowercase aur strip."""
    df = df.copy()
    df.columns = [str(c).strip().lower() for c in df.columns]
    return df
 
 
def is_empty(value) -> bool:
    """Null / empty / placeholder values check."""
    if value is None: return True
    s = str(value).strip()
    return s == "" or s.lower() in ("nan", "none", "null", "na", "n/a", "[]")
 
 
def is_garbage(value) -> bool:
    """Null byte ya invisible characters wali values."""
    s = str(value)
    if "\u0000" in s: return True
    return len(s.replace("\u0000", "").strip()) == 0
 
 
# ─── Core Type Checker ───────────────────────────────────────────────────────
 
def check_value_type(value, mapped_type: str) -> bool:
    """
    Value ko uske mapped_type ke against validate karo.
    Empty values yahan nahi aatein (pehle is_empty check hota hai).
    """
    val_str = str(value).strip()
 
    # ── BIGINT: sirf whole numbers ─────────────────────────────────────────────
    if mapped_type == "bigint":
        try:
            f = float(val_str)
            return f == int(f)   # 42.0 ✅ | 42.5 ❌
        except (ValueError, TypeError):
            return False
 
    # ── DOUBLE: koi bhi number ────────────────────────────────────────────────
    elif mapped_type == "double":
        try:
            float(val_str)       # scientific notation bhi handle hota hai
            return True
        except (ValueError, TypeError):
            return False
 
    # ── DECIMAL: high-precision via Python Decimal ────────────────────────────
    elif mapped_type == "decimal":
        try:
            Decimal(val_str)
            return True
        except (InvalidOperation, ValueError, TypeError):
            return False
 
    # ── STRING: koi bhi non-empty text ────────────────────────────────────────
    elif mapped_type == "string":
        return True   # Agar is_empty se guzar gaya hai toh valid hai
 
    # ── BOOLEAN ───────────────────────────────────────────────────────────────
    elif mapped_type == "boolean":
        return val_str.lower() in ("true", "false", "1", "0", "yes", "no")
 
    # ── DATE ──────────────────────────────────────────────────────────────────
    elif mapped_type == "date":
        try:
            parsed = pd.to_datetime(val_str, errors="raise")
            # Ensure karo ke yeh sirf date hai, time component nahi
            # Lekin agar Trino se aaya hai toh timestamp bhi date column mein ho sakta hai
            return True
        except Exception:
            return False
 
    # ── TIMESTAMP ─────────────────────────────────────────────────────────────
    elif mapped_type == "timestamp":
        try:
            pd.to_datetime(val_str, errors="raise")
            return True
        except Exception:
            return False
 
    # ── ARRAY<STRING> ─────────────────────────────────────────────────────────
    elif mapped_type == "array_string":
        try:
            # Value ek valid JSON/Python list honi chahiye
            parsed = ast.literal_eval(val_str)
            if not isinstance(parsed, list):
                return False
            # Empty list valid hai
            if len(parsed) == 0:
                return True
            # Har element string hona chahiye
            # NOTE: single-char strings bhi valid hain e.g. ["A", "B"]
            for item in parsed:
                if not isinstance(item, str):
                    return False
            return True
        except Exception:
            return False
 
    # ── Unknown mapped type — skip ─────────────────────────────────────────────
    return True
 
 
# ─── Detailed Validator (NULL / GARBAGE / TYPE ERROR classify karo) ───────────
 
def validate_value_detailed(value, mapped_type: str):
    """
    Returns:
        status (str): 'NULL' | 'GARBAGE' | 'TYPE_ERROR' | 'CLEAN'
        is_error (bool): True agar GARBAGE ya TYPE_ERROR ho
    """
    if is_empty(value):
        return "NULL", False
 
    if is_garbage(value):
        return "GARBAGE", True
 
    if not check_value_type(value, mapped_type):
        return "TYPE_ERROR", True
 
    return "CLEAN", False
 
 
# ─── Quick self-test ──────────────────────────────────────────────────────────
print("✅ Functions loaded! Type normalization test:")
test_types = [
    ("ARRAY<STRING>", "array_string"),
    ("BIGINT",        "bigint"),
    ("BOOLEAN",       "boolean"),
    ("DATE",          "date"),
    ("DOUBLE",        "double"),
    ("STRING",        "string"),
    ("TIMESTAMP",     "timestamp"),
    ("DECIMAL",       "decimal"),
    ("DECIMAL(38,18)","decimal"),
]
all_pass = True
for raw, expected in test_types:
    got = normalize_dtype(raw)
    ok  = "✅" if got == expected else "❌"
    if got != expected: all_pass = False
    print(f"  {ok} '{raw:20s}' → '{got}' (expected: '{expected}')")
print(f"\n{'✅ All type mappings correct!' if all_pass else '❌ Some mappings need fixing!'}")

✅ Functions loaded! Type normalization test:
  ✅ 'ARRAY<STRING>       ' → 'array_string' (expected: 'array_string')
  ✅ 'BIGINT              ' → 'bigint' (expected: 'bigint')
  ✅ 'BOOLEAN             ' → 'boolean' (expected: 'boolean')
  ✅ 'DATE                ' → 'date' (expected: 'date')
  ✅ 'DOUBLE              ' → 'double' (expected: 'double')
  ✅ 'STRING              ' → 'string' (expected: 'string')
  ✅ 'TIMESTAMP           ' → 'timestamp' (expected: 'timestamp')
  ✅ 'DECIMAL             ' → 'decimal' (expected: 'decimal')
  ✅ 'DECIMAL(38,18)      ' → 'decimal' (expected: 'decimal')

✅ All type mappings correct!


---
## 📂 Cell 5 — Schema Load aur Column Sync Check

In [11]:
# ── 1. Schema file load karo ─────────────────────────────────────────────────
try:
    schema_df = normalize_columns(read_file(SCHEMA_FILE))
    print("✅ Schema file loaded!")
except FileNotFoundError:
    print(f"❌ Schema file nahi mili: {SCHEMA_FILE}")
    raise

# ── 2. Schema structure validate karo ────────────────────────────────────────
dt_col        = DATATYPE_COLUMN_NAME.strip().lower()
required_cols = {"id", "field_name", dt_col}
missing_req   = required_cols - set(schema_df.columns)
if missing_req:
    print(f"❌ Schema file mein required columns nahi hain: {missing_req}")
    print(f"   Jo columns hain: {sorted(schema_df.columns.tolist())}")
    raise ValueError(f"Missing schema columns: {missing_req}")

# ── 3. Data normalize karo (Trino se jo aaya) ────────────────────────────────
data_df = normalize_columns(data_df)

print(f"\n✅ Schema rows    : {len(schema_df)}")
print(f"✅ Data rows      : {len(data_df)}")
print(f"✅ Data columns   : {len(data_df.columns)}")

# ── 4. Schema fields extract karo ────────────────────────────────────────────
schema_fields    = schema_df["field_name"].str.strip().str.lower().tolist()
schema_raw_types = schema_df[dt_col].str.strip().tolist()
field_type_map   = dict(zip(schema_fields, schema_raw_types))
data_columns     = list(data_df.columns)

schema_set = set(schema_fields)
data_set   = set(data_columns)

# ── 5. Column comparison ──────────────────────────────────────────────────────
matched_cols             = [c for c in data_columns if c in schema_set]
extra_in_data            = [c for c in data_columns if c not in schema_set]
missing_from_data        = [f for f in schema_fields if f not in data_set]

print("\n" + "=" * 70)
print("📊 COLUMN SYNCHRONIZATION SUMMARY")
print("=" * 70)
print(f"  Schema fields total          : {len(schema_fields)}")
print(f"  Data columns total           : {len(data_columns)}")
print(f"  ✅ Matched (will validate)    : {len(matched_cols)}")
print(f"  ⚠️  Extra in DATA (not schema) : {len(extra_in_data)}")
print(f"  ⚠️  In SCHEMA, not in DATA    : {len(missing_from_data)}")
print("=" * 70)

# ── ERROR: Data mein schema se zyada columns ──────────────────────────────────
if len(data_columns) > len(schema_fields):
    print(f"\n❌ ERROR: Data mein schema se zyada columns hain!")
    print(f"   Data: {len(data_columns)} columns | Schema: {len(schema_fields)} fields")
    print(f"   Extra columns in data: {extra_in_data}")
    print(f"\n   NOTE: Agar yeh spelling mismatch ho sakti hai toh neeche dekho:")

# ── Possible name mismatches (fuzzy hint) ─────────────────────────────────────
if extra_in_data or missing_from_data:
    print("\n📋 Columns not validated in DATA file:")
    for i, col in enumerate(extra_in_data, 1):
        print(f"   {i:3}. {col}")

    print(f"\n📋 Schema fields not found in DATA (count: {len(missing_from_data)}):")
    for i, col in enumerate(missing_from_data, 1):
        print(f"   {i:3}. {col}")

    # Possible spelling mismatches — jab dono mein similar names hon
    possible_mismatches = []
    for d_col in extra_in_data:
        for s_col in missing_from_data:
            # Simple check: ek mein doosra contained ho ya 80%+ characters match hon
            if d_col in s_col or s_col in d_col:
                possible_mismatches.append((d_col, s_col))
            elif len(d_col) > 3 and len(s_col) > 3:
                common = sum(c in s_col for c in d_col)
                ratio  = common / max(len(d_col), len(s_col))
                if ratio > 0.75:
                    possible_mismatches.append((d_col, s_col))

    if possible_mismatches:
        print("\n🔍 Possible name mismatches (Data → Schema):")
        for d, s in possible_mismatches:
            print(f"   Data: '{d}'  ←→  Schema: '{s}'")

print()

# ── Previews ─────────────────────────────────────────────────────────────────
print("=" * 55)
print("📄 SCHEMA — Pehli 5 rows")
print("=" * 55)
display(schema_df.head(5))

print(f"\n📊 DATA FROM TRINO — {data_df.shape[0]} rows × {data_df.shape[1]} columns")
display(data_df.head(5))

✅ Schema file loaded!

✅ Schema rows    : 144
✅ Data rows      : 439548
✅ Data columns   : 144

📊 COLUMN SYNCHRONIZATION SUMMARY
  Schema fields total          : 144
  Data columns total           : 144
  ✅ Matched (will validate)    : 144
  ⚠️  Extra in DATA (not schema) : 0
  ⚠️  In SCHEMA, not in DATA    : 0

📄 SCHEMA — Pehli 5 rows


,id,field_name,datatype
0,27533,ABCD,BIGINT
1,27534,Account,STRING
2,27535,AvgPx,"DECIMAL(38,18)"
3,27536,BoothID,STRING
4,27537,BoothIdent,STRING



📊 DATA FROM TRINO — 439548 rows × 144 columns


,completeleaveqty,s3_path,complianceid,origclordid,sendercompid,lastpx,symbolwithoutsfx,id,statusdesc,optionsymbol,...,lastshares,locationid,messagetag,coveredoruncovered,versionmaj,orderdate,status,contrabroker,key,account
0,500.000000000000000000,s3a://kuza-ignite/bronze/AuditTrailDAS/AWS/Pri...,NaN,NaN,NaN,NaN,IFS,BTG59510000001,NaN,NaN,...,NaN,NaN,68,NaN,3,2026-04-22,2026-04-22T13:30:24.156,NaN,BTG59510000001,VCLT-LAT-3BTGCC-D
1,500.000000000000000000,s3a://kuza-ignite/bronze/AuditTrailDAS/AWS/Pri...,NaN,NaN,NaN,NaN,IFS,BTG-4273-1-2,NaN,NaN,...,NaN,NaN,8,NaN,4,2026-04-22,2026-04-22T13:30:24.156,NaN,BTG-4273-1-2,VCLT-LAT-3BTGCC-D
2,500.000000000000000000,s3a://kuza-ignite/bronze/AuditTrailDAS/AWS/Pri...,NaN,NaN,NaN,NaN,IFS,BTG59510000002,NaN,NaN,...,NaN,NaN,68,NaN,3,2026-04-22,2026-04-22T13:36:05.198,NaN,BTG59510000002,VCLT-LAT-3BTGCC-D
3,500.000000000000000000,s3a://kuza-ignite/bronze/AuditTrailDAS/AWS/Pri...,NaN,NaN,NaN,NaN,IFS,BTG-4275-1-2,NaN,NaN,...,NaN,NaN,8,NaN,4,2026-04-22,2026-04-22T13:36:05.198,NaN,BTG-4275-1-2,VCLT-LAT-3BTGCC-D
4,500.000000000000000000,s3a://kuza-ignite/bronze/AuditTrailDAS/AWS/Pri...,NaN,NaN,NaN,NaN,IFS,BTG59510000003,NaN,NaN,...,NaN,NaN,68,NaN,3,2026-04-22,2026-04-22T13:40:27.108,NaN,BTG59510000003,VCLT-LAT-3BTGCC-D


---
## ✅ Cell 6 — Validation Run karo

In [12]:
column_results = {}
all_error_rows = []
print("=" * 80)
print("🔍 IGNITE DATA LAKE: DETAILED VALIDATION REPORT")
print("=" * 80)
print(f"{'COLUMN':<25} | {'SCHEMA TYPE':<16} | {'NULLS':>5} | {'GARBAGE':>7} | {'TYPE ERR':>8} | STATUS")
print("-" * 80)

for field in matched_cols:
    raw_dtype   = field_type_map[field]
    mapped_type = normalize_dtype(raw_dtype)

    # Unknown type → skip
    if mapped_type is None:
        print(f"{field[:25]:<25} | {raw_dtype[:16]:<16} | {'':>5} | {'':>7} | {'':>8} | ⚠️  SKIPPED (unknown type)")
        column_results[field] = {
            "status"          : "skipped",
            "errors"          : [],
            "raw_type"        : raw_dtype,
            "null_count"      : 0,
            "garbage_count"   : 0,
            "type_error_count": 0,
            "null_pct"        : 0.0,
            "high_null"       : False,
        }
        continue

    null_c, garb_c, type_c = 0, 0, 0
    field_errors = []

    for row_idx, val in enumerate(data_df[field], start=2):
        status, is_err = validate_value_detailed(val, mapped_type)
        if   status == "NULL"      : null_c += 1
        elif status == "GARBAGE"   : garb_c += 1; field_errors.append({"Row": row_idx, "Column": field, "Schema Type": raw_dtype, "Issue": "GARBAGE",    "Value": val})
        elif status == "TYPE_ERROR": type_c += 1; field_errors.append({"Row": row_idx, "Column": field, "Schema Type": raw_dtype, "Issue": "TYPE_ERROR", "Value": val})

    # ── 90%+ NULL highlight ──────────────────────────────────────────────
    total_rows = len(data_df[field])
    null_pct   = (null_c / total_rows * 100) if total_rows > 0 else 0
    high_null  = null_pct >= 90
    # ────────────────────────────────────────────────────────────────────

    total_errs = garb_c + type_c

    if high_null and total_errs == 0:
        status_str = f"⚠️  HIGH NULL ({null_pct:.0f}%) ✅ PASS"
    elif high_null and total_errs > 0:
        status_str = f"⚠️  HIGH NULL ({null_pct:.0f}%) ❌ FAIL"
    elif total_errs == 0:
        status_str = "✅ PASS"
    else:
        status_str = "❌ FAIL"

    print(f"{field[:25]:<25} | {raw_dtype[:16]:<16} | {null_c:>5} | {garb_c:>7} | {type_c:>8} | {status_str}")

    column_results[field] = {
        "status"          : "pass" if total_errs == 0 else "fail",
        "errors"          : field_errors,
        "raw_type"        : raw_dtype,
        "null_count"      : null_c,
        "garbage_count"   : garb_c,
        "type_error_count": type_c,
        "null_pct"        : round(null_pct, 1),
        "high_null"       : high_null,
    }
    all_error_rows.extend(field_errors)

print("=" * 80)
passed  = sum(1 for v in column_results.values() if v["status"] == "pass")
failed  = sum(1 for v in column_results.values() if v["status"] == "fail")
skipped = sum(1 for v in column_results.values() if v["status"] == "skipped")
high_null_cols = sum(1 for v in column_results.values() if v.get("high_null"))

print(f"  ✅ PASS       : {passed}")
print(f"  ❌ FAIL       : {failed}")
print(f"  ⚠️  SKIPPED    : {skipped}")
print(f"  ⚠️  HIGH NULL  : {high_null_cols}")
print(f"  Total flagged rows: {len(all_error_rows)}")
print("=" * 80)

🔍 IGNITE DATA LAKE: DETAILED VALIDATION REPORT
COLUMN                    | SCHEMA TYPE      | NULLS | GARBAGE | TYPE ERR | STATUS
--------------------------------------------------------------------------------
completeleaveqty          | DECIMAL(38,18)   |     0 |       0 |        0 | ✅ PASS
s3_path                   | STRING           |     0 |       0 |        0 | ✅ PASS
complianceid              | STRING           | 439548 |       0 |        0 | ⚠️  HIGH NULL (100%) ✅ PASS
origclordid               | STRING           | 66285 |       0 |        0 | ✅ PASS
sendercompid              | STRING           | 225814 |       0 |        0 | ✅ PASS
lastpx                    | DECIMAL(38,18)   | 427874 |       0 |        0 | ⚠️  HIGH NULL (97%) ✅ PASS
symbolwithoutsfx          | STRING           |     0 |       0 |        0 | ✅ PASS
id                        | STRING           |     0 |       0 |        0 | ✅ PASS
statusdesc                | STRING           | 439548 |       0 |        0 | ⚠️  

---
## 📋 Cell 7 — Error Detail Table

In [7]:
if all_error_rows:
    error_df = pd.DataFrame(all_error_rows)
    print(f"❌ Total {len(error_df)} errors:\n")
    display(error_df)
else:
    print("✅ Koi error nahi! Data clean hai.")

✅ Koi error nahi! Data clean hai.


---
## 📊 Cell 8 — Column-wise Summary Table

In [8]:
summary_rows = []
for field, info in column_results.items():
    emoji = {"pass": "✅ PASS", "fail": "❌ FAIL", "skipped": "⚠️ SKIPPED"}.get(info["status"], "?")
    summary_rows.append({
        "Column"          : field,
        "Schema Type"     : info["raw_type"],
        "Mapped Type"     : normalize_dtype(info["raw_type"]) or "unknown",
        "NULL Count"      : info["null_count"],
        "Garbage Count"   : info["garbage_count"],
        "Type Error Count": info["type_error_count"],
        "Status"          : emoji,
    })

summary_df = pd.DataFrame(summary_rows)
print("📊 Column-wise Validation Summary:")
display(summary_df)

📊 Column-wise Validation Summary:


,Column,Schema Type,Mapped Type,NULL Count,Garbage Count,Type Error Count,Status
0,completeleaveqty,DECIMAL,decimal,0,0,0,✅ PASS
1,s3_path,STRING,string,0,0,0,✅ PASS
2,origclordid,STRING,string,7823,0,0,✅ PASS
3,sendercompid,STRING,string,5463,0,0,✅ PASS
4,lastpx,"DECIMAL(38,18)",decimal,5290,0,0,✅ PASS
...,...,...,...,...,...,...,...
93,versionmaj,BIGINT,bigint,0,0,0,✅ PASS
94,orderdate,DATE,date,0,0,0,✅ PASS
95,status,STRING,string,0,0,0,✅ PASS
96,key,STRING,string,0,0,0,✅ PASS


---
## 🔎 Cell 9 — Kisi Specific Column Ki Detail Dekho (Optional)

In [ ]:
# Kisi bhi column ki values inspect karne ke liye yahan naam daalo
DEBUG_COLUMN = "marketlookid"   # <-- yahan apna column naam daalo
MAX_SHOW     = 20               # Kitni rows dikhani hain

if DEBUG_COLUMN not in data_df.columns:
    print(f"❌ Column '{DEBUG_COLUMN}' data mein nahi hai.")
else:
    raw_dtype   = field_type_map.get(DEBUG_COLUMN, "UNKNOWN")
    mapped_type = normalize_dtype(raw_dtype)
    print(f"Column       : {DEBUG_COLUMN}")
    print(f"Schema Type  : {raw_dtype} → Mapped: {mapped_type}")
    print()

    rows = []
    for idx, val in enumerate(data_df[DEBUG_COLUMN], start=2):
        status, _ = validate_value_detailed(val, mapped_type)
        rows.append({"Row": idx, "Value": val, "Status": status})

    debug_df = pd.DataFrame(rows)
    counts   = debug_df["Status"].value_counts()
    print(f"Status breakdown:\n{counts.to_string()}\n")

    # Errors dikhao
    errors_only = debug_df[debug_df["Status"].isin(["TYPE_ERROR", "GARBAGE"])]
    if not errors_only.empty:
        print(f"❌ Errors (first {MAX_SHOW}):")
        display(errors_only.head(MAX_SHOW))
    else:
        print("✅ Is column mein koi error nahi.")